# 07 Demo UI

**Owner.** Jack (Product Lead & Knowledge Architect) · **Supports.** Tim

**Goal.** A lawyer can use the agent without seeing code: type a case, get an answer with sources, and see when the
agent was unsure. The same interface is used for the live demo on 23 November.

**Definition of Done.** The UI calls the real pipeline and shows answer, sources, area and guard status.
Screenshot in the log, backup video recorded (tasks T15, T26, T27).

## 1. What the user must see

- **Answer** structured as norm, subsumption, result.
- **Sources** as links to the articles, so every statement can be checked in one click.
- **Status:** area of law, number of reviewer rounds, whether a human should look at it, whether personal data was removed.
- **Nothing else.** No prompts, no tokens, no settings. Lawyers judge the tool by whether they trust it.

## 2. Connecting the pipeline

`run_pipeline` is the only function the UI calls. Today it returns a fixed example so the layout can be built in
parallel. In W4 it calls the input guard from 05, the graph from 04 and the citation check from 05.

In [ ]:
def run_pipeline(question):
    """Placeholder. Replace with: input_guard -> legal_agent.invoke -> citation_check."""
    return {
        "answer": "**Normwahl.** Art. 336c Abs. 1 lit. b OR.\n\n"
                  "**Subsumtion.** Im dritten Dienstjahr gilt bei Krankheit eine Sperrfrist von 90 Tagen.\n\n"
                  "**Ergebnis.** Eine Kündigung während dieser Zeit ist nichtig.",
        "sources": [{"id": "OR-336c", "title": "Kündigung zur Unzeit",
                     "url": "https://www.fedlex.admin.ch/eli/cc/27/317_321_377/de#art_336_c"}],
        "area": "arbeitsrecht",
        "revisions": 0,
        "needs_human": False,
        "guard": "ok",
        "placeholder": True,
    }

## 3. The interface

In [ ]:
import gradio as gr

EXAMPLES = [
    "Ich arbeite seit drei Jahren in einer Firma und bin seit zwei Wochen krank. Mein Chef will mir kündigen. Darf er das?",
    "Ich habe zwei Monatsmieten nicht bezahlt. Was kann der Vermieter tun?",
    "In meiner Wohnung ist Schimmel. Darf ich die Miete reduzieren?",
]

def answer_case(question):
    result = run_pipeline(question)
    sources = "\n".join(f"- [{s['id']}: {s['title']}]({s['url']})" for s in result["sources"]) or "Keine Quellen."
    status = (f"**Rechtsgebiet:** {result['area']} · **Prüfrunden:** {result['revisions']} · "
              f"**Guard:** {result['guard']}")
    if result["needs_human"]:
        status += "\n\n⚠️ Der Agent ist unsicher. Bitte durch eine Fachperson prüfen lassen."
    if result.get("placeholder"):
        status += "\n\n_Platzhalter-Antwort: Pipeline noch nicht verbunden._"
    return result["answer"], sources, status

with gr.Blocks(title="Swiss Legal Agent") as demo:
    gr.Markdown("# Swiss Legal Agent\nArbeitsrecht und Mietrecht nach OR. Prototyp aus dem Unterricht, keine Rechtsberatung.")
    question = gr.Textbox(label="Sachverhalt", lines=4, placeholder="Beschreiben Sie den Fall ...")
    ask = gr.Button("Prüfen", variant="primary")
    gr.Examples(EXAMPLES, inputs=question)
    with gr.Row():
        answer = gr.Markdown(label="Antwort")
        with gr.Column(scale=1):
            sources = gr.Markdown(label="Quellen")
            status = gr.Markdown(label="Status")
    ask.click(answer_case, inputs=question, outputs=[answer, sources, status])

## 4. Launching in Deepnote

Deepnote does not show Gradio inline. `share=True` creates a public link that is valid for 72 hours, which is also
how we open it on the presentation laptop. For the demo itself we record a backup video (T27) in case the network fails.

In [1]:
# T15: the UI lives in the repository (lexgraph/ui/app.py), so notebook, demo and tests run the same code.
import pathlib, subprocess, sys
REPO_ROOT = pathlib.Path("swiss_legal_agent").resolve()
sys.path.insert(0, str(REPO_ROOT))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_ROOT / "requirements.txt")], check=True)
from lexgraph.ui.app import demo
demo.launch(share=True)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
deepnote-toolkit 2.9.1 requires filelock<5,>=4.2.1; python_version >= "3.11", but you have filelock 3.32.7 which is incompatible.
deepnote-toolkit 2.9.1 requires ipykernel<7,>=6.30.1, but you have ipykernel 7.3.0 which is incompatible.
deepnote-toolkit 2.9.1 requires pillow<14,>=13.0.0; python_version >= "3.11", but you have pillow 12.3.0 which is incompatible.

[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
/root/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
* Running on local URL:  http://127.0.0.1:7860


KeyboardInterrupt: 

## Your turn

- Show the sources before the answer. Does that change how much you trust the answer?
- What should the UI show when the input guard blocks a question?
- Ask someone outside the team to use the UI without any explanation. Where do they hesitate?

## Decision log

| Date | Decision | Rejected alternative | Reason |
|---|---|---|---|
| Oct 2026 | Gradio | Streamlit | Covered in the course, runs inside the notebook, shareable link. |
| | Sources shown as links to Fedlex | | |
| | Demo via share link plus backup video | | |

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=9b9d17b7-b4fd-473b-beb2-f7ed180fbdd3' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>